# World Happiness Data Analysis

**Authors:** Sofia Jordan, Laure Reytan, and Alexia Lazes  
**Date:** December 2025  
**Format:** Three-person academic project

## Research question

How has reported happiness changed across countries and over time, and how is it associated with economic, social, institutional, and emotional factors?

This notebook uses the World Happiness Report 2024 dataset to clean and explore 2,363 country-year observations, construct an exploratory composite score, examine correlations, and estimate simple OLS models. The analysis is descriptive: the relationships identified here should not be interpreted as causal effects.


## Dataset

The dataset is available on [Kaggle](https://www.kaggle.com/datasets/muskanmaheshwari15/world-happiness-data-2024) and covers country-year observations through 2023.

Key variables include:

- **Life Ladder:** reported life satisfaction
- **Log GDP per capita:** economic prosperity
- **Social support:** perceived access to supportive relationships
- **Healthy life expectancy:** expected years lived in good health
- **Freedom to make life choices:** perceived autonomy
- **Generosity:** charitable giving and prosocial behaviour
- **Perceptions of corruption:** perceived public-sector corruption
- **Positive and negative affect:** reported emotional experiences

To reproduce the analysis, download the CSV and save it as `data/World_Happiness_Report_2024.csv`.


## 1. Setup and data loading


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_PATH = Path("data/World_Happiness_Report_2024.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download it from the Kaggle link in the Dataset section."
    )

df_raw = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df_raw):,} | Columns: {df_raw.shape[1]}")
df_raw.head()


## 2. Data cleaning


In [ ]:
df = df_raw.copy()

df = df.rename(
    columns={
        "Country name": "Country",
        "year": "Year",
        "Log GDP per capita": "GDP",
        "Healthy life expectancy at birth": "Life expectancy",
        "Freedom to make life choices": "Freedom",
        "Perceptions of corruption": "Corruption",
    }
)

analysis_columns = [
    "Life Ladder",
    "GDP",
    "Social support",
    "Life expectancy",
    "Freedom",
    "Generosity",
    "Corruption",
    "Positive affect",
    "Negative affect",
]

print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values before cleaning:")
display(df[analysis_columns].isna().sum().sort_values(ascending=False))

# Mean imputation keeps the full panel for this exploratory analysis.
# A production analysis should compare alternative imputation strategies.
for column in analysis_columns:
    df[column] = df[column].fillna(df[column].mean())

print("Missing values after cleaning:", int(df[analysis_columns].isna().sum().sum()))


### Cleaning note

Mean imputation is transparent and reproducible, but it reduces variability and can weaken relationships in the data. The results should therefore be treated as exploratory rather than definitive.


## 3. Descriptive statistics


In [ ]:
summary = df[analysis_columns].agg(["mean", "median", "std"]).T
summary.columns = ["Mean", "Median", "Standard deviation"]
summary.round(3)


In [ ]:
latest_year = int(df["Year"].max())
df_latest = df.loc[df["Year"] == latest_year].copy()

happiest = df_latest.nlargest(5, "Life Ladder")[["Country", "Life Ladder"]]
lowest = df_latest.nsmallest(5, "Life Ladder")[["Country", "Life Ladder"]]

print(f"Highest reported life satisfaction in {latest_year}")
display(happiest)
print(f"Lowest reported life satisfaction in {latest_year}")
display(lowest)


## 4. Exploratory composite score


To compare several dimensions on a common scale, the variables are min-max normalized. Corruption and negative affect are inverted so that higher values consistently represent more favourable conditions.

The resulting score is an **equally weighted exploratory index**, not a validated measure of national welfare. Its purpose is to support comparison, not to replace the Life Ladder measure.


In [ ]:
normalized = df[["Country", "Year"] + analysis_columns].copy()

for column in analysis_columns:
    minimum = df[column].min()
    maximum = df[column].max()
    normalized[column] = (df[column] - minimum) / (maximum - minimum)

normalized["Low corruption"] = 1 - normalized["Corruption"]
normalized["Low negative affect"] = 1 - normalized["Negative affect"]

score_columns = [
    "Life Ladder",
    "GDP",
    "Social support",
    "Life expectancy",
    "Freedom",
    "Generosity",
    "Positive affect",
    "Low corruption",
    "Low negative affect",
]

normalized["Overall score"] = normalized[score_columns].mean(axis=1)
normalized[["Life Ladder", "Overall score"]].describe().round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normalized["Life Ladder"], bins=30, alpha=0.65, label="Life Ladder")
ax.hist(normalized["Overall score"], bins=30, alpha=0.65, label="Exploratory overall score")
ax.set(title="Life Ladder and exploratory composite score", xlabel="Normalized score", ylabel="Frequency")
ax.legend()
plt.show()


In [ ]:
yearly_score = normalized.groupby("Year", as_index=False)["Overall score"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(yearly_score["Year"], yearly_score["Overall score"], marker="o")
ax.set(title="Average exploratory score over time", xlabel="Year", ylabel="Average score")
plt.show()


## 5. Emotional experience and life satisfaction


In [ ]:
corr_positive = df_latest["Positive affect"].corr(df_latest["Life Ladder"])
corr_negative = df_latest["Negative affect"].corr(df_latest["Life Ladder"])

print(f"Positive affect correlation: {corr_positive:.3f}")
print(f"Negative affect correlation: {corr_negative:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(df_latest["Positive affect"], df_latest["Life Ladder"], alpha=0.7)
axes[0].set(title="Positive affect", xlabel="Positive affect", ylabel="Life Ladder")
axes[1].scatter(df_latest["Negative affect"], df_latest["Life Ladder"], alpha=0.7, color="tab:red")
axes[1].set(title="Negative affect", xlabel="Negative affect")
plt.tight_layout()
plt.show()


In [ ]:
df_latest["Emotional balance"] = (
    df_latest["Positive affect"] - df_latest["Negative affect"]
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df_latest["Emotional balance"], bins=25, edgecolor="white")
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set(title=f"Emotional balance across countries in {latest_year}",
       xlabel="Positive affect minus negative affect", ylabel="Number of countries")
plt.show()


In [ ]:
affect_over_time = df.groupby("Year", as_index=False)[
    ["Positive affect", "Negative affect"]
].mean()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(affect_over_time["Year"], affect_over_time["Positive affect"], marker="o", label="Positive affect")
ax.plot(affect_over_time["Year"], affect_over_time["Negative affect"], marker="o", label="Negative affect")
ax.set(title="Average global affect over time", xlabel="Year", ylabel="Average score")
ax.legend()
plt.show()


## 6. Economic, social, and institutional correlates


In [ ]:
factors = [
    "GDP",
    "Social support",
    "Life expectancy",
    "Freedom",
    "Generosity",
    "Corruption",
    "Positive affect",
    "Negative affect",
]

correlations = (
    df_latest[factors + ["Life Ladder"]]
    .corr()["Life Ladder"]
    .drop("Life Ladder")
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 5))
correlations.plot.barh(ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title=f"Correlation with life satisfaction in {latest_year}", xlabel="Pearson correlation")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df_latest["GDP"], df_latest["Life Ladder"], alpha=0.7, color="tab:green")
axes[0].set(title="GDP and life satisfaction", xlabel="Log GDP per capita", ylabel="Life Ladder")
axes[1].scatter(df_latest["Social support"], df_latest["Life Ladder"], alpha=0.7, color="tab:blue")
axes[1].set(title="Social support and life satisfaction", xlabel="Social support", ylabel="Life Ladder")
plt.tight_layout()
plt.show()


## 7. Single-factor OLS models


The models below estimate each factor separately using normalized values. This makes coefficient magnitudes easier to compare, but the estimates remain **unadjusted associations**. They do not establish that a factor causes changes in happiness, and omitted variables may influence both the predictor and the outcome.


In [ ]:
model_factors = ["GDP", "Social support", "Life expectancy", "Freedom", "Generosity", "Corruption"]
results = []

for factor in model_factors:
    model_data = normalized[["Life Ladder", factor]].dropna()
    y = model_data["Life Ladder"]
    X = sm.add_constant(model_data[factor])
    model = sm.OLS(y, X).fit()
    results.append(
        {
            "Factor": factor,
            "Coefficient": model.params[factor],
            "P-value": model.pvalues[factor],
            "R-squared": model.rsquared,
            "Observations": int(model.nobs),
        }
    )

model_results = pd.DataFrame(results).sort_values("R-squared", ascending=False)
model_results.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(model_results["Factor"], model_results["R-squared"])
ax.invert_yaxis()
ax.set(title="Explanatory power of single-factor models", xlabel="R-squared", ylabel="")
plt.show()


## 8. Conclusions and limitations

The analysis shows that life satisfaction varies substantially across countries and is associated with a combination of economic conditions, social support, health, institutional perceptions, freedom, and emotional experience. GDP and social support display particularly strong relationships with life satisfaction in the single-factor models, while positive and negative affect move in the expected directions.

These findings should be interpreted cautiously:

- Correlation and OLS association do not demonstrate causation.
- Country-year observations are not fully independent.
- Mean imputation reduces variability in variables with missing data.
- The exploratory composite score uses equal weights chosen for transparency, not validated welfare weights.
- National averages can conceal substantial differences within countries.

Future work could use panel-data methods, regional comparisons, alternative imputation strategies, and out-of-sample validation.
